In [3]:
# ============================================================
# 1. IMPORTS
# ============================================================

import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

print("Imports loaded successfully")

Imports loaded successfully


In [4]:
# ============================================================
# 2. PROJECT PATH AND INTERNAL MODULES
# ============================================================

project_root = Path(r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data_pipeline
import src.events

importlib.reload(src.data_pipeline)
importlib.reload(src.events)

from src.data_pipeline import ModelDatasetBuilder
from src.events import SimpleEventDetector, CombinedEventDetector

print("Internal modules loaded successfully")

Internal modules loaded successfully


In [5]:
# ============================================================
# 3. DATABASE PATH AND BUILDER
# ============================================================

db_path = project_root / "NordPoool" / "data" / "thesis_database.db"

print("DB exists:", db_path.exists())
print("DB path:", db_path)

builder = ModelDatasetBuilder(db_path)

print("Builder created successfully")

DB exists: True
DB path: C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\NordPoool\data\thesis_database.db
Builder created successfully


In [6]:
# ============================================================
# 4. LOAD DATA FOR LSTM
# ============================================================

df = builder.build_price_dataset(
    zones="NO1",
    start_date="2020-01-01",
    end_date="2020-12-31",
    add_time_features=True,
    lags=[1, 2, 24, 48, 168],
    target_horizon=1,
    include_volumes=True,
    dropna=False
)

print("Dataset shape:", df.shape)
print("Start:", df.index.min())
print("End:", df.index.max())

df.head()

Dataset shape: (8761, 17)
Start: 2020-01-01 00:00:00
End: 2020-12-31 00:00:00


,price_id,zone_id,delivery_day,hour,price_value,buy_volume_value,sell_volume_value,year,month,day,day_of_week,price_value_lag_1,price_value_lag_2,price_value_lag_24,price_value_lag_48,price_value_lag_168,target
datetime,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,32,12,2020-01-01,0,31.77,4091.8,1819.6,2020,1,1,2,NaN,NaN,NaN,NaN,NaN,31.57
2020-01-01 01:00:00,52,12,2020-01-01,1,31.57,4021.3,1826.2,2020,1,1,2,31.77,NaN,NaN,NaN,NaN,31.28
2020-01-01 02:00:00,72,12,2020-01-01,2,31.28,3975.7,1836.8,2020,1,1,2,31.57,31.77,NaN,NaN,NaN,30.72
2020-01-01 03:00:00,92,12,2020-01-01,3,30.72,3993.6,1841.5,2020,1,1,2,31.28,31.57,NaN,NaN,NaN,30.27
2020-01-01 04:00:00,112,12,2020-01-01,4,30.27,4041.5,1798.0,2020,1,1,2,30.72,31.28,NaN,NaN,NaN,30.17


In [7]:
# ============================================================
# 5. PRICE, VOLUME AND COMBINED EVENT DETECTION
# ============================================================

detector = SimpleEventDetector()
combined_detector = CombinedEventDetector()

# Price events
df_events = detector.detect_price_events(df)

# Volume events
df_events = detector.detect_volume_events(df_events)

# Combined events
df_events = combined_detector.detect_combined_events(df_events)

print("Dataset with all events shape:", df_events.shape)

# Check generated event columns
event_cols_check = [
    # Price events
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility",

    # Volume events
    "high_demand",
    "low_demand",
    "high_generation",
    "low_generation",
    "volume_imbalance",
    "strong_buy_pressure",
    "strong_sell_pressure",
    "buy_volume_spike",
    "sell_volume_spike",

    # Combined events
    "generation_surplus",
    "demand_pressure",
    "strong_demand_pressure",
    "strong_generation_pressure",
    "demand_driven_price_spike",
    "generation_driven_low_price",
    "scarcity_price_event",
    "oversupply_price_event",
    "buy_pressure_price_spike",
    "sell_pressure_low_price",
    "price_separation",
    "extreme_price_separation",
    "zone_price_outlier",
    "has_combined_event",
]

existing_event_cols = [col for col in event_cols_check if col in df_events.columns]

df_events[existing_event_cols].head()

Dataset with all events shape: (8761, 60)


,low_price,high_price,price_spike,extreme_price,rapid_price_change,price_ramp_up,price_ramp_down,high_volatility,high_demand,low_demand,...,demand_driven_price_spike,generation_driven_low_price,scarcity_price_event,oversupply_price_event,buy_pressure_price_spike,sell_pressure_low_price,price_separation,extreme_price_separation,zone_price_outlier,has_combined_event
datetime,,,,,,,,,,,,,,,,,,,,,
2020-01-01 00:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 01:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 02:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 03:00:00,False,True,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2020-01-01 04:00:00,False,True,True,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
# ============================================================
# 7. CYCLICAL TIME FEATURES
# ============================================================

df_events = df_events.copy()

df_events["hour"] = df_events.index.hour
df_events["dayofweek"] = df_events.index.dayofweek

df_events["hour_sin"] = np.sin(2 * np.pi * df_events["hour"] / 24)
df_events["hour_cos"] = np.cos(2 * np.pi * df_events["hour"] / 24)

df_events["dayofweek_sin"] = np.sin(2 * np.pi * df_events["dayofweek"] / 7)
df_events["dayofweek_cos"] = np.cos(2 * np.pi * df_events["dayofweek"] / 7)

df_events[
    [
        "hour",
        "hour_sin",
        "hour_cos",
        "dayofweek",
        "dayofweek_sin",
        "dayofweek_cos"
    ]
].head()

,hour,hour_sin,hour_cos,dayofweek,dayofweek_sin,dayofweek_cos
datetime,,,,,,
2020-01-01 00:00:00,0,0.000000,1.000000,2,0.974928,-0.222521
2020-01-01 01:00:00,1,0.258819,0.965926,2,0.974928,-0.222521
2020-01-01 02:00:00,2,0.500000,0.866025,2,0.974928,-0.222521
2020-01-01 03:00:00,3,0.707107,0.707107,2,0.974928,-0.222521
2020-01-01 04:00:00,4,0.866025,0.500000,2,0.974928,-0.222521


In [9]:
# ============================================================
# 8. DEFINE LSTM FEATURE SETS
# ============================================================

price_base_features_lstm = [
    "price_value",

    # Calendar features
    "year",
    "month",
    "day",
    "day_of_week",

    # Cyclical time features
    "hour_sin",
    "hour_cos",
    "dayofweek_sin",
    "dayofweek_cos",
]

lag_features_lstm = [
    "price_value_lag_1",
    "price_value_lag_2",
    "price_value_lag_24",
    "price_value_lag_48",
    "price_value_lag_168",
]

volume_base_features_lstm = [
    "buy_volume_value",
    "sell_volume_value",
]

price_event_features_lstm = [
    "low_price",
    "high_price",
    "price_spike",
    "extreme_price",
    "rapid_price_change",
    "price_ramp_up",
    "price_ramp_down",
    "high_volatility",
]

volume_event_features_lstm = [
    "high_demand",
    "low_demand",
    "high_generation",
    "low_generation",
    "volume_imbalance",
    "strong_buy_pressure",
    "strong_sell_pressure",
    "buy_volume_delta",
    "sell_volume_delta",
    "abs_buy_volume_delta",
    "abs_sell_volume_delta",
    "buy_volume_spike",
    "sell_volume_spike",
]

combined_event_features_lstm = [
    "generation_surplus",
    "demand_pressure",
    "strong_demand_pressure",
    "strong_generation_pressure",
    "demand_driven_price_spike",
    "generation_driven_low_price",
    "scarcity_price_event",
    "oversupply_price_event",
    "buy_pressure_price_spike",
    "sell_pressure_low_price",
    "price_separation",
    "extreme_price_separation",
    "zone_price_outlier",
    "has_combined_event",
]

lstm_feature_sets = {
    "baseline_price_lags": (
        price_base_features_lstm
        + lag_features_lstm
    ),

    "baseline_price_lags_volume": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
    ),

    "price_events": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
        + price_event_features_lstm
    ),

    "price_volume_events": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
        + price_event_features_lstm
        + volume_event_features_lstm
    ),

    "all_events": (
        price_base_features_lstm
        + lag_features_lstm
        + volume_base_features_lstm
        + price_event_features_lstm
        + volume_event_features_lstm
        + combined_event_features_lstm
    ),
}

target_col = "target"

# Check missing columns
for set_name, cols in lstm_feature_sets.items():
    missing_cols = [col for col in cols if col not in df_events.columns]
    print(set_name, "n_features:", len(cols), "missing:", missing_cols)

baseline_price_lags n_features: 14 missing: []
baseline_price_lags_volume n_features: 16 missing: []
price_events n_features: 24 missing: []
price_volume_events n_features: 37 missing: []
all_events n_features: 51 missing: []


In [10]:
# ============================================================
# 9. TEMPORAL TRAIN / TEST SPLIT DATES
# ============================================================

train_end = "2020-11-30 23:00:00"
test_start = "2020-12-01 00:00:00"
test_end = "2020-12-08 00:00:00"

lookback = 24

print("Train end:", train_end)
print("Test period:", test_start, "to", test_end)
print("LSTM lookback:", lookback, "hours")

Train end: 2020-11-30 23:00:00
Test period: 2020-12-01 00:00:00 to 2020-12-08 00:00:00
LSTM lookback: 24 hours


In [11]:
# ============================================================
# 10. CREATE LSTM SEQUENCES FUNCTION
# ============================================================

def create_lstm_sequences(X, y, lookback):
    X_seq = []
    y_seq = []
    seq_index = []

    for i in range(lookback, len(X)):
        X_seq.append(X.iloc[i - lookback:i].values)
        y_seq.append(y.iloc[i])
        seq_index.append(y.index[i])

    X_seq = np.array(X_seq)
    y_seq = np.array(y_seq)
    seq_index = pd.Index(seq_index)

    return X_seq, y_seq, seq_index

print("Function created successfully")

Function created successfully


In [12]:
# ============================================================
# 11. FUNCTION TO TRAIN AND EVALUATE LSTM BY FEATURE SET
# ============================================================

def evaluate_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


def train_evaluate_lstm_feature_set(feature_set_name, feature_cols, lookback=24):

    print("=" * 70)
    print(f"Training LSTM feature set: {feature_set_name}")
    print("=" * 70)

    # Prepare dataset
    df_lstm = df_events[feature_cols + [target_col]].dropna().copy()

    X_raw = df_lstm[feature_cols]
    y_raw = df_lstm[target_col]

    # Temporal masks
    train_mask = df_lstm.index <= train_end
    test_mask = (df_lstm.index >= test_start) & (df_lstm.index <= test_end)

    X_train_raw = X_raw.loc[train_mask]
    y_train_raw = y_raw.loc[train_mask]

    print("Raw X:", X_raw.shape)
    print("Raw y:", y_raw.shape)
    print("Train:", X_train_raw.shape)
    print("Number of features:", len(feature_cols))

    # Scaling fitted only on train
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    scaler_X.fit(X_train_raw)
    scaler_y.fit(y_train_raw.values.reshape(-1, 1))

    X_scaled = pd.DataFrame(
        scaler_X.transform(X_raw),
        index=X_raw.index,
        columns=X_raw.columns
    )

    y_scaled = pd.Series(
        scaler_y.transform(y_raw.values.reshape(-1, 1)).ravel(),
        index=y_raw.index,
        name=target_col
    )

    # Create LSTM sequences
    X_seq, y_seq, seq_index = create_lstm_sequences(
        X_scaled,
        y_scaled,
        lookback
    )

    train_seq_mask = seq_index <= train_end
    test_seq_mask = (seq_index >= test_start) & (seq_index <= test_end)

    X_train = X_seq[train_seq_mask]
    y_train = y_seq[train_seq_mask]

    X_test = X_seq[test_seq_mask]
    y_test = y_seq[test_seq_mask]

    y_test_index = seq_index[test_seq_mask]

    print("X_train_lstm:", X_train.shape)
    print("X_test_lstm:", X_test.shape)

    # Define LSTM model
    n_timesteps = X_train.shape[1]
    n_features = X_train.shape[2]

    model = Sequential([
        LSTM(
            units=64,
            input_shape=(n_timesteps, n_features),
            return_sequences=False
        ),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    # Train
    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        shuffle=False,
        verbose=0
    )

    # Predict
    y_pred_scaled = model.predict(X_test, verbose=0).ravel()

    # Inverse scaling
    y_pred = scaler_y.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()

    y_test_real = scaler_y.inverse_transform(
        y_test.reshape(-1, 1)
    ).ravel()

    pred_series = pd.Series(
        y_pred,
        index=y_test_index,
        name=f"LSTM_{feature_set_name}"
    )

    real_series = pd.Series(
        y_test_real,
        index=y_test_index,
        name="Real"
    )

    metrics = evaluate_model(real_series, pred_series)

    result = {
        "feature_set": feature_set_name,
        "model": f"LSTM_{lookback}h",
        "n_features": len(feature_cols),
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
        "epochs_trained": len(history.history["loss"])
    }

    return model, history, real_series, pred_series, result

In [13]:
# ============================================================
# 12. TRAIN LSTM FOR ALL FEATURE SETS
# ============================================================

lstm_models_by_features = {}
lstm_histories_by_features = {}
lstm_real_by_features = {}
lstm_predictions_by_features = {}
lstm_results = []

for feature_set_name, feature_cols in lstm_feature_sets.items():

    model, history, real_series, pred_series, result = train_evaluate_lstm_feature_set(
        feature_set_name=feature_set_name,
        feature_cols=feature_cols,
        lookback=lookback
    )

    lstm_models_by_features[feature_set_name] = model
    lstm_histories_by_features[feature_set_name] = history
    lstm_real_by_features[feature_set_name] = real_series
    lstm_predictions_by_features[feature_set_name] = pred_series
    lstm_results.append(result)

    print("Finished:", feature_set_name)
    print(result)

Training LSTM feature set: baseline_price_lags
Raw X: (8592, 14)
Raw y: (8592,)
Train: (7872, 14)
Number of features: 14
X_train_lstm: (7848, 24, 14)
X_test_lstm: (169, 24, 14)


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: baseline_price_lags
{'feature_set': 'baseline_price_lags', 'model': 'LSTM_24h', 'n_features': 14, 'MAE': 1.6775433139688163, 'RMSE': np.float64(2.35070540776376), 'R2': 0.4687200056510913, 'epochs_trained': 34}
Training LSTM feature set: baseline_price_lags_volume
Raw X: (8592, 16)
Raw y: (8592,)
Train: (7872, 16)
Number of features: 16
X_train_lstm: (7848, 24, 16)
X_test_lstm: (169, 24, 16)


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: baseline_price_lags_volume
{'feature_set': 'baseline_price_lags_volume', 'model': 'LSTM_24h', 'n_features': 16, 'MAE': 1.9109911030425122, 'RMSE': np.float64(2.5263639006586183), 'R2': 0.38635264885450515, 'epochs_trained': 38}
Training LSTM feature set: price_events
Raw X: (8592, 24)
Raw y: (8592,)
Train: (7872, 24)
Number of features: 24
X_train_lstm: (7848, 24, 24)
X_test_lstm: (169, 24, 24)


...

Finished: price_events
{'feature_set': 'price_events', 'model': 'LSTM_24h', 'n_features': 24, 'MAE': 2.112661147484413, 'RMSE': np.float64(2.744856620654054), 'R2': 0.27562012304814576, 'epochs_trained': 39}
Training LSTM feature set: price_volume_events
Raw X: (8592, 37)
Raw y: (8592,)
Train: (7872, 37)
Number of features: 37
X_train_lstm: (7848, 24, 37)
X_test_lstm: (169, 24, 37)


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: price_volume_events
{'feature_set': 'price_volume_events', 'model': 'LSTM_24h', 'n_features': 37, 'MAE': 1.9671425265250118, 'RMSE': np.float64(2.590341052352378), 'R2': 0.3548793456490661, 'epochs_trained': 32}
Training LSTM feature set: all_events
Raw X: (8592, 51)
Raw y: (8592,)
Train: (7872, 51)
Number of features: 51
X_train_lstm: (7848, 24, 51)
X_test_lstm: (169, 24, 51)


c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished: all_events
{'feature_set': 'all_events', 'model': 'LSTM_24h', 'n_features': 51, 'MAE': 1.7459486544908154, 'RMSE': np.float64(2.2718402048287802), 'R2': 0.5037704622767367, 'epochs_trained': 40}


In [14]:
# ============================================================
# 13. LSTM FEATURE SET COMPARISON
# ============================================================

lstm_feature_comparison_df = pd.DataFrame(lstm_results)

lstm_feature_comparison_df = lstm_feature_comparison_df.sort_values(
    ["RMSE", "MAE"],
    ascending=True
)

lstm_feature_comparison_df

,feature_set,model,n_features,MAE,RMSE,R2,epochs_trained
4,all_events,LSTM_24h,51,1.745949,2.271840,0.503770,40
0,baseline_price_lags,LSTM_24h,14,1.677543,2.350705,0.468720,34
1,baseline_price_lags_volume,LSTM_24h,16,1.910991,2.526364,0.386353,38
3,price_volume_events,LSTM_24h,37,1.967143,2.590341,0.354879,32
2,price_events,LSTM_24h,24,2.112661,2.744857,0.275620,39
